# 环节 06 · 凭证与密钥管理演示

纯 Python 标准库，零依赖。手搓四件事：

1. 泄漏窗口 = TTL × 暴露面；
2. 四种方案（env / deny / mask / 代签）能力矩阵；
3. mask 链路（哨兵替换 + fail-closed）；
4. SigV4 类凭证能否重签。

In [ ]:
# §1 泄漏窗口 = TTL × 暴露面
def window_hours(ttl_hours, scope_factor):
    return ttl_hours * scope_factor


SCENARIOS = [
    ("长期 Key + 全账号权限", 24 * 365, 1.00),
    ("长期 Key + 单桶只读",   24 * 365, 0.05),
    ("1 小时 STS + 单桶只读", 1,        0.05),
    ("5 分钟 one-shot + 单对象写", 5 / 60, 0.001),
]
print(f"{'方案':<30}{'TTL':<10}{'暴露面':<10}{'风险量级'}")
print("-" * 66)
for name, ttl, scope in SCENARIOS:
    print(f"{name:<30}{ttl:<10.4g}{scope:<10.2f}{window_hours(ttl, scope):.4g}")
print()
print("→ 两个因子都要压：短 TTL 与最小 scope")

In [ ]:
# §2 四方案能力矩阵
PLANS = {
    "env/文件注入": dict(can_use=True,  agent_sees_plain=True),
    "deny":         dict(can_use=False, agent_sees_plain=False),
    "mask":         dict(can_use=True,  agent_sees_plain=False),
    "代理代签":     dict(can_use=True,  agent_sees_plain=False),
}
print(f"{'方案':<14}{'Agent 能用':<12}{'Agent 看到明文':<15}{'评价'}")
print("-" * 62)
NOTE = {
    "env/文件注入": "反模式：一条 env/printenv 就带走",
    "deny":         "读不到也用不了（需另找通道）",
    "mask":         "能用但拿不到；需 TLS 终止",
    "代理代签":     "最干净：真值不出代理",
}
for name, p in PLANS.items():
    print(f"{name:<14}{str(p['can_use']):<12}{str(p['agent_sees_plain']):<15}{NOTE[name]}")

In [ ]:
# §3 mask 链路模拟：哨兵 -> 出站替换（含 fail-closed）
SENTINEL = "sk-SENTINEL-0000"
REAL = "sk-real-secret-abcdef"
INJECT_HOSTS = ["api.github.com"]


def sandbox_env():
    return {"GH_TOKEN": SENTINEL}       # 沙箱内只看到哨兵


def egress_proxy(request, tls_terminate):
    host, body = request["host"], request["body"]
    masked = SENTINEL in body
    if not masked:
        return body, "原样转发（无哨兵，无需替换）"
    if not tls_terminate:
        return body, "FAIL-CLOSED：未终止 TLS，哨兵原样发出 → 认证失败"
    if host not in INJECT_HOSTS:
        return body, "目标主机不在 injectHosts → 不替换"
    return body.replace(SENTINEL, REAL), "已替换为真值（仅在出站瞬间）"


env = sandbox_env()
print("沙箱内进程看到:", env)
print()
req = {"host": "api.github.com", "body": f"Authorization: Bearer {SENTINEL}"}
for tls in [False, True]:
    out, how = egress_proxy(req, tls)
    print(f"tlsTerminate={tls!s:<5} → {how}")
print()
print("若目标主机不在 allowedDomains 里，请求根本到不了代理 → 永远注入不了")

In [ ]:
# §4 哪些凭证能"整串替换"，哪些必须重签
CREDS = {
    "普通 API Key":  dict(kind="静态串",    resign=False, ok=True),
    "JWT":           dict(kind="结构化",    resign=False, ok=True),
    "AWS SigV4":     dict(kind="内容相关签名", resign=True,  ok=True),
    "aws-chunked":   dict(kind="流式链式签名", resign=False, ok=False),
    "预签名 URL":     dict(kind="URL 内签名",  resign=False, ok=False),
    "SigV4A":        dict(kind="非对称签名",   resign=False, ok=False),
}
print(f"{'凭证形态':<16}{'类型':<14}{'需重签':<8}{'可安全代签'}")
print("-" * 52)
for name, c in CREDS.items():
    print(f"{name:<16}{c['kind']:<14}{str(c['resign']):<8}{'可以' if c['ok'] else '需透传/拒绝'}")
print()
print("→ 『能不能给 Agent 云权限』取决于凭证形态，选型时是硬指标")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 为什么凭证不该进模型上下文？ | 会被复述/落日志/转发，且无法撤回 |
| 2 | deny 与 mask 的差别？ | deny 读不到也用不了；mask 能用但拿不到真值 |
| 3 | mask 为什么必须 TLS 终止？ | 代理要在明文请求里替换；否则 fail-closed |
| 4 | SigV4 为什么不能简单替换 AccessKey？ | 签名与请求内容/时间绑定，替换即失效 |
| 5 | 哪些请求代理无法重签？ | aws-chunked、预签名 URL、SigV4A |
| 6 | 沙箱被完全攻破时最后一道防线？ | 代理侧最小权限 + 短 TTL + 审计 |

**相关长文**：[环节06-凭证与密钥管理详解.md](./环节06-凭证与密钥管理详解.md)